# E7 step 2b · Is the chain's gap to R11 on the test pairs more than noise? (your PC, CPU)

**Why.** E7 step 2's rule (roadmap G6, fixed before that run) reopened the choice of chain: on E6's 2,007 clean test pairs, the destroyed-building IoU is 0.3898 for the chain (SAM 3 finds each building, a grader labels it; the mean of three grader seeds) against 0.3984 for R11 (the U-Net that reads the before and after photos, with test-time augmentation). Before choosing, this checks whether a gap of 0.0086 is larger than the variation that comes from which pairs happen to be in the test set.

**How.**
1. Score every method on every clean pair from step 2's saved files, against E6's true masks: R11 and grader seed 42 from their saved maps; seeds 43 and 44 rebuilt from SAM 3's outlines (`sam3_pre_grid2.zip`) and each grader's scores for each outline (`instances_test.csv`, with R16c's fixed biases). The totals must reproduce step 2's published scores: exactly for R11 and seed 42, and within 0.001 for seeds 43 and 44, whose scores were saved rounded to 6 decimals.
2. Resample 10,000 times and recompute the difference *chain (mean of the three graders) minus R11* in destroyed-building IoU:
   - **by pair:** the 2,007 pairs drawn with replacement;
   - **by event:** the 28 disaster events drawn with replacement, each with all its pairs, because pairs from one event share their conditions (place, sensor, damage type), so they aren't independent.
3. Also for the pairs without Turkey, and for Turkey alone (by pair only: one event can't be resampled by event).

**How to read it.** If the middle 95% of the resampled differences includes 0, the gap is within noise.

**Expectation, fixed before the run (11 Nov):** on all clean pairs, the 95% interval includes 0 at both levels, since the gap is small next to the differences between events. Without Turkey, R11's lead (0.057) holds by pair. On Turkey alone, the chain's lead (0.24) holds. The intervals by event are wider than by pair.

**First run (11 Nov):** it stopped at Cell 3's check, before any resampling: rebuilt from the rounded scores, seed 43 on Turkey gave 0.3293 against the published 0.3299 (every other number within 0.0001). Seed 42 is now scored from its saved maps, and seeds 43 and 44 get a tolerance of 0.001, with the near-ties counted. The expectation above was not changed.

Nothing here is tuned on the test pairs: it only measures how certain step 2's published numbers are.


In [1]:
# ── Cell 1: Inputs and helpers ───────────────────────────────────────────────
import io, json, re, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

STEP2 = Path("E7-results/step2")
E6 = Path("E6-results/E6")
OUT = Path("E7-results/step2b")
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 43, 44)
LABELS = ["Intact", "Damaged", "Destroyed", "None"]
BIASES = {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}   # step 2's BIAS_FIXED
METHODS = [f"chain_s{s}" for s in SEEDS] + ["R11_tta"]
RESAMPLES, RNG_SEED = 10_000, 0

def event_of(name):
    """E1's rule, as in step 2."""
    m = re.match(r"^(.+?)_(\d{8})_post_disaster", name)
    if m:
        return m.group(1)
    m = re.match(r"^(.+?)_post", name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem)

def read_png(z, name, shape=None):
    a = np.array(Image.open(io.BytesIO(z.read(name))))
    if shape is not None and a.shape != shape:
        a = np.array(Image.fromarray(a).resize((shape[1], shape[0]), Image.NEAREST))
    return a

def confusion(gt, pred):
    """4x4 pixel counts, rows = truth, columns = prediction (step 2's function)."""
    idx = gt.astype(np.int64).ravel() * 4 + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=16).reshape(4, 4)

def class_iou(cm, c):
    tp = cm[..., c, c]
    return tp / (cm[..., c, :].sum(-1) + cm[..., :, c].sum(-1) - tp + 1e-8)

pairs = pd.read_csv(E6 / "test_pairs.csv")
clean = pairs[pairs["clean"] == True].reset_index(drop=True)
assert len(clean) == 2007, len(clean)
clean["event_e1"] = clean["after_image"].map(event_of)
inst = pd.read_csv(STEP2 / "instances_test.csv")
published = pd.read_csv(STEP2 / "map_scores.csv")
print(f"✓ {len(clean):,} clean test pairs, {clean['event_e1'].nunique()} events; {len(inst):,} SAM 3 outlines with grader scores")


✓ 2,007 clean test pairs, 28 events; 138,725 SAM 3 outlines with grader scores


In [2]:
# ── Cell 2: Rebuild the maps and count pixels per pair (a few minutes) ───────
# Seed 42's maps are scored from the saved files (exactly as step 2 scored them). Seeds 43 and 44 have no saved maps, so they're
# rebuilt from the graders' scores, which step 2 saved rounded to 6 decimals: an outline whose two best labels were within about
# 1e-6 can flip label. Rebuilding seed 42 the same way measures how much that changes (first run, 11 Nov: the totals matched
# step 2 within 0.0006).
t0 = time.time()
z_sam, z_r11, z_s42 = (zipfile.ZipFile(STEP2 / f) for f in ("sam3_pre_grid2.zip", "r11_tta_maps.zip", "chain_s42_maps.zip"))
z_gt = zipfile.ZipFile(E6 / "test_masks_4class.zip")
gt_names = set(z_gt.namelist())
by_image = {name: g for name, g in inst.groupby("image")}
CM = np.zeros((len(clean), len(METHODS), 4, 4), np.int64)
s42_mismatch_px, s42_px, resized = 0, 0, 0
for k, name in enumerate(clean["after_image"]):
    r11 = read_png(z_r11, name)
    shape = r11.shape
    gt = read_png(z_gt, name if name in gt_names else clean.loc[k, "mask_file"], shape)
    ids = read_png(z_sam, Path(name).stem + ".png").astype(np.int32)
    if ids.shape != shape:
        resized += 1
        ids = read_png(z_sam, Path(name).stem + ".png", shape).astype(np.int32)
    g = by_image.get(name)
    for j, seed in enumerate(SEEDS):
        lut = np.zeros(int(ids.max()) + 1, np.uint8)
        if g is not None:
            lp = g[[f"logp_{l.lower()}_s{seed}" for l in LABELS]].to_numpy() + np.array(BIASES[seed])
            lab = lp.argmax(1)
            lut[g["id"].to_numpy()] = np.where(lab < 3, lab + 1, 0)     # intact, damaged, destroyed -> 1-3; none -> 0
        pred = lut[ids]
        if seed == 42:
            saved = read_png(z_s42, name, shape)
            s42_mismatch_px += int((pred != saved).sum())
            s42_px += saved.size
            pred = saved                                                # seed 42 exactly as step 2 scored it
        CM[k, j] = confusion(gt, pred)
    CM[k, 3] = confusion(gt, r11)
print(f"✓ Rebuilt {len(clean):,} pairs in {(time.time() - t0) / 60:.1f} min; SAM 3 maps resized: {resized}")
print(f"  Rebuilding seed 42 from the rounded scores changes {s42_mismatch_px:,} of {s42_px:,} pixels "
      f"({100 * s42_mismatch_px / s42_px:.4f}%); seed 42 is scored from its saved maps")


✓ Rebuilt 2,007 pairs in 3.1 min; SAM 3 maps resized: 0
  Rebuilding seed 42 from the rounded scores changes 9,393 of 2,104,492,032 pixels (0.0004%); seed 42 is scored from its saved maps


In [3]:
# ── Cell 3: The totals must reproduce step 2's published scores ──────────────
# R11 and seed 42 come from saved maps, so they must match to the 4 published decimals. Seeds 43 and 44 are rebuilt from rounded
# scores (Cell 2), so they may differ by a little: up to 0.001 is allowed, and the near-ties that explain it are counted.
SCOPES = {"clean pairs": lambda e: True, "clean pairs, without Turkey": lambda e: not e.startswith("turkey"),
          "Turkey only": lambda e: e.startswith("turkey")}
checks = []
for scope, keep in SCOPES.items():
    rows = clean["event_e1"].map(keep).to_numpy()
    for j, m in enumerate(METHODS):
        cm = CM[rows, j].sum(0)
        pub = published[(published["scope"] == scope) & (published["method"] == m)].iloc[0]
        ours = {"destroyed_iou": round(float(class_iou(cm, 3)), 4), "miou": round(float(np.mean([class_iou(cm, c) for c in range(4)])), 4)}
        checks.append({"scope": scope, "method": m, **{f"{k}_rebuilt": v for k, v in ours.items()},
                       **{f"{k}_published": float(pub[k]) for k in ours}})
checks = pd.DataFrame(checks)
checks["largest_difference"] = np.maximum((checks["destroyed_iou_rebuilt"] - checks["destroyed_iou_published"]).abs(),
                                          (checks["miou_rebuilt"] - checks["miou_published"]).abs()).round(4)
print(checks.to_string(index=False))
near_ties = {}
for seed in SEEDS:
    lp = inst[[f"logp_{l.lower()}_s{seed}" for l in LABELS]].to_numpy() + np.array(BIASES[seed])
    top2 = np.sort(lp, axis=1)[:, -2:]
    near_ties[seed] = int((top2[:, 1] - top2[:, 0] < 2e-6).sum())
print(f"Outlines whose two best labels are within 2e-6 (can flip when rebuilt from 6-decimal scores), by seed: {near_ties}")
exact = checks["method"].isin(["R11_tta", "chain_s42"])
assert checks.loc[exact, "largest_difference"].max() < 5e-5, "R11 or seed 42 doesn't reproduce step 2 from its saved maps"
assert checks.loc[~exact, "largest_difference"].max() <= 0.001, "seeds 43 and 44 differ from step 2 by more than rounding explains"
print(f"✓ Step 2's published scores reproduced: R11 and seed 42 exactly; seeds 43 and 44 within "
      f"{checks.loc[~exact, 'largest_difference'].max():.4f}")


                      scope    method  destroyed_iou_rebuilt  miou_rebuilt  destroyed_iou_published  miou_published  largest_difference
                clean pairs chain_s42                 0.4055        0.5179                   0.4055          0.5179              0.0000
                clean pairs chain_s43                 0.3982        0.5121                   0.3983          0.5121              0.0001
                clean pairs chain_s44                 0.3657        0.5086                   0.3657          0.5087              0.0001
                clean pairs   R11_tta                 0.3984        0.5721                   0.3984          0.5721              0.0000
clean pairs, without Turkey chain_s42                 0.4230        0.5263                   0.4230          0.5263              0.0000
clean pairs, without Turkey chain_s43                 0.4189        0.5173                   0.4189          0.5173              0.0000
clean pairs, without Turkey chain_s44           

In [4]:
# ── Cell 4: Resampling: the chain (mean of three) minus R11, destroyed-building IoU ──
TP = CM[:, :, 3, 3].astype(np.float64)                                          # (pairs, methods)
UNION = (CM[:, :, 3, :].sum(-1) + CM[:, :, :, 3].sum(-1)).astype(np.float64) - TP

def gap(w, tp, union):
    """w: (resamples, units) weights -> chain mean-of-three minus R11, per resample."""
    iou = (w @ tp) / (w @ union + 1e-8)                                        # (resamples, methods)
    return iou[:, :3].mean(1) - iou[:, 3]

rng = np.random.default_rng(RNG_SEED)
results = []
for scope, keep in SCOPES.items():
    rows = clean["event_e1"].map(keep).to_numpy()
    tp, union, ev = TP[rows], UNION[rows], clean.loc[rows, "event_e1"].to_numpy()
    observed = float(gap(np.ones((1, len(tp))), tp, union)[0])
    levels = {"by pair": (tp, union)}
    events = sorted(set(ev))
    if len(events) > 1:
        idx = np.array([events.index(e) for e in ev])
        levels["by event"] = (np.stack([tp[idx == i].sum(0) for i in range(len(events))]),
                              np.stack([union[idx == i].sum(0) for i in range(len(events))]))
    for level, (t, u) in levels.items():
        n = len(t)
        diffs = np.concatenate([gap(rng.multinomial(n, np.full(n, 1 / n), size=1000).astype(np.float64), t, u)
                                for _ in range(RESAMPLES // 1000)])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        results.append({"scope": scope, "level": level, "units": n, "observed_gap": round(observed, 4),
                        "ci95_low": round(float(lo), 4), "ci95_high": round(float(hi), 4),
                        "share_chain_ahead_%": round(100 * float((diffs > 0).mean()), 1),
                        "zero_inside_ci": bool(lo <= 0 <= hi)})
results = pd.DataFrame(results)
print(results.to_string(index=False))


                      scope    level  units  observed_gap  ci95_low  ci95_high  share_chain_ahead_%  zero_inside_ci
                clean pairs  by pair   2007       -0.0086   -0.0378     0.0218                 28.1            True
                clean pairs by event     28       -0.0086   -0.0978     0.0894                 38.3            True
clean pairs, without Turkey  by pair   1593       -0.0567   -0.0779    -0.0363                  0.0           False
clean pairs, without Turkey by event     27       -0.0567   -0.1114    -0.0204                  0.0           False
                Turkey only  by pair    414        0.2364    0.1614     0.3040                100.0           False


In [5]:
# ── Cell 5: Against the expectation, and the output files ────────────────────
def row(scope, level):
    return results[(results["scope"] == scope) & (results["level"] == level)].iloc[0]

expect = [
    ("all clean pairs, by pair: the interval includes 0", row("clean pairs", "by pair")["zero_inside_ci"]),
    ("all clean pairs, by event: the interval includes 0", row("clean pairs", "by event")["zero_inside_ci"]),
    ("without Turkey, by pair: R11's lead holds (interval below 0)", row("clean pairs, without Turkey", "by pair")["ci95_high"] < 0),
    ("Turkey only, by pair: the chain's lead holds (interval above 0)", row("Turkey only", "by pair")["ci95_low"] > 0),
    ("the intervals by event are wider than by pair",
     all((row(s, "by event")["ci95_high"] - row(s, "by event")["ci95_low"]) > (row(s, "by pair")["ci95_high"] - row(s, "by pair")["ci95_low"])
         for s in ("clean pairs", "clean pairs, without Turkey"))),
]
for text, ok in expect:
    print(f"  {'met' if ok else 'NOT met'}: {text}")

per_pair = clean[["after_image", "event_e1"]].copy()
for j, m in enumerate(METHODS):
    per_pair[f"{m}_destroyed_tp"] = TP[:, j].astype(np.int64)
    per_pair[f"{m}_destroyed_union"] = UNION[:, j].astype(np.int64)
per_pair.to_csv(OUT / "per_pair_destroyed.csv", index=False)
results.to_csv(OUT / "resampling.csv", index=False)
checks.to_csv(OUT / "reproduction_check.csv", index=False)

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

main = row("clean pairs", "by pair"), row("clean pairs", "by event")
within = main[0]["zero_inside_ci"] and main[1]["zero_inside_ci"]
(OUT / "README.md").write_text("\n".join([
    "# E7 step 2b: is the chain's gap to R11 on the test pairs more than noise?", "",
    "Run on the PC from E7 step 2's saved files ([step 2](../step2/README.md)); notebook `e7-step2b-resampling-test.ipynb`.", "",
    f"**On all 2,007 clean test pairs, the chain's destroyed-building IoU (mean of three graders) minus R11's is {main[0]['observed_gap']}: "
    + ("within noise. " if within else "more than noise. ")
    + f"The middle 95% of resampled gaps runs from {main[0]['ci95_low']} to {main[0]['ci95_high']} by pair, "
      f"and from {main[1]['ci95_low']} to {main[1]['ci95_high']} by event.**", "",
    "## Resampled gaps (chain minus R11, destroyed-building IoU)", "", as_markdown(results), "",
    "- `by pair`: the clean pairs drawn with replacement, 10,000 times. `by event`: the disaster events drawn with replacement, each "
    "with all its pairs, because pairs from one event aren't independent.",
    "- `share_chain_ahead_%`: the share of resamples in which the chain's score is higher.", "",
    "## Against the expectation, fixed before the run", "",
    *[f"- {'Met' if ok else 'Not met'}: {text}." for text, ok in expect], "",
    "## Checks", "",
    "- R11 and grader seed 42 are scored from step 2's saved maps and reproduce its published destroyed IoU and mIoU exactly "
    "(`reproduction_check.csv`).",
    f"- Seeds 43 and 44 have no saved maps, so they're rebuilt from the graders' scores, which step 2 saved rounded to 6 decimals; an "
    f"outline whose two best labels were nearly tied can flip. They match step 2 within {checks.loc[~exact, 'largest_difference'].max():.4f} "
    f"(outlines within 2e-6, by seed: {near_ties}). Rebuilding seed 42 the same way changes {100 * s42_mismatch_px / s42_px:.4f}% of its "
    "pixels. This is far below the resampled spread.", "",
    "## Files", "", "- `resampling.csv`, `reproduction_check.csv`, `per_pair_destroyed.csv` (each pair's destroyed-class "
    "intersection and union for every method)",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT}: README.md, resampling.csv, reproduction_check.csv, per_pair_destroyed.csv")


  met: all clean pairs, by pair: the interval includes 0
  met: all clean pairs, by event: the interval includes 0
  met: without Turkey, by pair: R11's lead holds (interval below 0)
  met: Turkey only, by pair: the chain's lead holds (interval above 0)
  met: the intervals by event are wider than by pair
✓ Wrote E7-results\step2b: README.md, resampling.csv, reproduction_check.csv, per_pair_destroyed.csv
